# Paying Attention — Try it in PyTorch

This is an **optional** hands-on companion to [Chapter 7: Paying Attention](https://learnai.robennals.org/attention). I'll build single-head attention from scratch using the chapter's exact 4-token toy vocab (cat, dog, blah, it), implement softmax from scratch, extend with values, then peek inside real **BERT** to see what its actual attention heads have learned. Finally I'll see how the Q/K/V vectors come from `nn.Linear` projections of the embedding.

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

## Building Attention from Scratch

I'll start with the chapter's tiny toy: a 4-token vocabulary (cat, dog, blah, it) where each token has a 1D key ("what I'm advertising") and a 1D query ("what I'm looking for"). The key for nouns is 3, for filler tokens 0. The query is 3 for "it" (looking for nouns) and 0 for everything else. Attention scores come from the dot product of one token's query with every other token's key, then softmax to turn scores into percentages.

In [ ]:
TOKEN_KEY = {'cat': 3.0, 'dog': 3.0, 'blah': 0.0, 'it': 0.0}
TOKEN_QUERY = {'cat': 0.0, 'dog': 0.0, 'blah': 0.0, 'it': 3.0}

print(f"{'token':6s} {'key':>5s} {'query':>5s}")
print('-' * 20)
for tok in TOKEN_KEY:
    print(f'{tok:6s} {TOKEN_KEY[tok]:5.1f} {TOKEN_QUERY[tok]:5.1f}')

In [ ]:
def softmax(scores):
    """Numerically-stable softmax."""
    scores = torch.tensor(scores, dtype=torch.float32)
    m = scores.max()
    exps = torch.exp(scores - m)
    return (exps / exps.sum()).tolist()

def attention_step(sentence, focus_token_index):
    """Compute attention from sentence[focus_token_index] to every token."""
    focus = sentence[focus_token_index]
    q = TOKEN_QUERY[focus]
    print(f'\n--- attention from {focus!r} (position {focus_token_index}) ---')
    print(f"  query = {q}")

    print(f"\n  {'pos':>3s}  {'token':6s} {'key':>4s} {'q*k':>5s}")
    scores = []
    for i, tok in enumerate(sentence):
        k = TOKEN_KEY[tok]
        score = q * k  # dot product (1D)
        scores.append(score)
        print(f'  {i:3d}  {tok:6s} {k:4.1f} {score:5.1f}')

    weights = softmax(scores)
    print(f"\n  attention weights (after softmax):")
    for tok, w in zip(sentence, weights):
        bar = '#' * int(w * 40)
        print(f'    {tok:6s} {w:.3f}  {bar}')
    return weights

# Chapter examples — focus is always 'it' (the last token in each)
for sent in [
    ['cat', 'blah', 'blah', 'it'],
    ['cat', 'blah', 'dog', 'it'],
    ['blah', 'blah', 'blah', 'it'],
]:
    attention_step(sent, len(sent) - 1)

## Softmax: Dividing Your Attention

Softmax turns arbitrary scores into positive numbers that add up to 100%. The key dynamic: **relative loudness wins**. A moderately-high score wins by default if everyone else is silent; a much-louder score drowns out a moderately-loud one; making all scores equally large changes nothing. Below I run the chapter's four scenarios.

In [ ]:
scenarios = [
    ('All equal',                [1.0, 1.0, 1.0, 1.0]),
    ('Moderate, rest silent',    [2.0, 0.0, 0.0, 0.0]),
    ('Moderate meets loud',      [2.0, 6.0, 0.0, 0.0]),
    ('All large but equal',      [8.0, 8.0, 8.0, 8.0]),
]

for label, scores in scenarios:
    weights = softmax(scores)
    print(f'\n--- {label} ---')
    print(f'  scores:  {scores}')
    print(f'  weights: {[round(w, 3) for w in weights]}')
    print(f'  sum:     {sum(weights):.4f}')

## Values: What Did You Find?

Attention so far tells us *where to look*. The third vector — the **value** — is *what gets said*. I'll give each toy token a value vector that encodes which animal it represents (cat = [1, 0], dog = [0, 1], filler and it = [0, 0]). Then I blend the values using the attention weights.

In [ ]:
TOKEN_VALUE = {
    'cat':  torch.tensor([1.0, 0.0]),
    'dog':  torch.tensor([0.0, 1.0]),
    'blah': torch.tensor([0.0, 0.0]),
    'it':   torch.tensor([0.0, 0.0]),
}

def attention_with_values(sentence, focus_index):
    focus = sentence[focus_index]
    q = TOKEN_QUERY[focus]
    scores = [q * TOKEN_KEY[tok] for tok in sentence]
    weights = softmax(scores)
    blended = torch.zeros(2)
    for tok, w in zip(sentence, weights):
        blended += w * TOKEN_VALUE[tok]
    print(f'\nFocus token: {focus!r} (position {focus_index})')
    for tok, w in zip(sentence, weights):
        print(f'  {tok:6s} weight={w:.3f}  value={TOKEN_VALUE[tok].tolist()}')
    print(f'  blended value: {blended.tolist()}  (cat-ness {blended[0]:.3f}, dog-ness {blended[1]:.3f})')

for sent in [
    ['cat', 'blah', 'blah', 'it'],
    ['cat', 'blah', 'dog', 'it'],
    ['blah', 'blah', 'blah', 'it'],
]:
    attention_with_values(sent, len(sent) - 1)

## Multi-Headed Attention

The toy mechanism above only solves one task — figuring out which noun a pronoun refers to. Real understanding means tracking many kinds of relationships at once: which adjective modifies which noun, where one clause ends and the next begins, what an earlier word is a callback to, whether a word is the topic or new information. A single set of keys and queries can only ask one kind of question.

Real models fix this by running many attention computations in parallel. Each one is called an **attention head**, with its own keys, queries, and values, free to learn its own pattern. The model gets all kinds of attention at once. I'll see this directly in BERT below.

## Attention in a Real Model

Toy attention with 1D scores is fine for getting the mechanism. **BERT** (`bert-base-uncased`) has 144 attention heads in total — every one learned its pattern from scratch during training, without being told what to look for. Most heads do something too subtle to describe in words, but a few follow patterns we can recognize. I'll load BERT (~440MB on first download), run a sentence through it, and extract attention from four heads with humanly-recognizable patterns: one looks at the next word, one at the previous word, one resolves pronouns, and one spreads broadly across context.

In [ ]:
from transformers import BertTokenizer, BertModel

print('Loading BERT (downloads ~440MB on first run)...')
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased', attn_implementation='eager')
bert_model.eval()
print('Loaded.')

SENTENCE = 'The dog chased the cat because it was angry'
inputs = bert_tokenizer(SENTENCE, return_tensors='pt')
tokens = bert_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print(f'\nTokens: {tokens}')

with torch.no_grad():
    outputs = bert_model(**inputs, output_attentions=True)

# attentions is a tuple of 12 tensors (one per layer), each (batch, n_heads, seq, seq)
print(f'\nAttentions: {len(outputs.attentions)} layers, '
      f'each tensor shape {outputs.attentions[0].shape}')

In [ ]:
# The 4 heads the chapter visualizes — same indices as scripts/extract-bert-attention.py
INTERESTING_HEADS = [
    (2, 0,  'Next word'),
    (3, 5,  'Previous word'),
    (4, 3,  'Self / pronoun'),
    (6, 11, 'Broad context'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, (layer, head, label) in zip(axes.flat, INTERESTING_HEADS):
    attn = outputs.attentions[layer][0, head].numpy()  # (seq, seq)
    ax.imshow(attn, cmap='Blues', aspect='auto')
    ax.set_title(f'Layer {layer} / Head {head}: {label}')
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticklabels(tokens)
    ax.set_xlabel('attended to')
    ax.set_ylabel('attending from')
plt.tight_layout()
plt.show()

# Spot-check the 'Self / pronoun' head from the perspective of 'it'
it_index = tokens.index('it')
layer, head = 4, 3
attn_from_it = outputs.attentions[layer][0, head, it_index].numpy()
print(f"\n'Self / pronoun' head — attention from 'it':")
for tok, w in zip(tokens, attn_from_it):
    bar = '#' * int(w * 40)
    print(f'  {tok:10s} {w:.3f}  {bar}')

## Where the Vectors Come From

Real attention doesn't have hand-picked 1D keys and queries. Each token starts as an embedding, and the Q/K/V vectors come from feeding that embedding through three separate single-layer neural networks (`nn.Linear` with no activation function). Below I demonstrate that property: give the same Q/K/V projections to similar embeddings (cat, dog) and they produce similar Q/K/V; give them a very different embedding (it) and the projections come out very different.

In [ ]:
EMBED_DIM = 4
PROJ_DIM = 3

# Synthetic embeddings — cat and dog are similar; 'it' is very different
EMBEDS = {
    'cat': torch.tensor([1.0, 0.0, 0.1, 0.0]),
    'dog': torch.tensor([0.9, 0.1, 0.0, 0.05]),
    'it':  torch.tensor([0.0, 0.0, 0.5, 0.8]),
}

torch.manual_seed(0)
Q_proj = nn.Linear(EMBED_DIM, PROJ_DIM, bias=False)
K_proj = nn.Linear(EMBED_DIM, PROJ_DIM, bias=False)
V_proj = nn.Linear(EMBED_DIM, PROJ_DIM, bias=False)

with torch.no_grad():
    for tok, emb in EMBEDS.items():
        q = Q_proj(emb)
        k = K_proj(emb)
        v = V_proj(emb)
        print(f'\n{tok:5s} embedding={emb.tolist()}')
        print(f'         Q={[round(x.item(), 3) for x in q]}')
        print(f'         K={[round(x.item(), 3) for x in k]}')
        print(f'         V={[round(x.item(), 3) for x in v]}')

# Confirm cat and dog produce similar Q vectors
with torch.no_grad():
    q_cat = Q_proj(EMBEDS['cat'])
    q_dog = Q_proj(EMBEDS['dog'])
    q_it  = Q_proj(EMBEDS['it'])
    sim_cat_dog = F.cosine_similarity(q_cat.unsqueeze(0), q_dog.unsqueeze(0)).item()
    sim_cat_it  = F.cosine_similarity(q_cat.unsqueeze(0), q_it.unsqueeze(0)).item()
print(f'\ncos(Q(cat), Q(dog)) = {sim_cat_dog:.3f}')
print(f'cos(Q(cat), Q(it))  = {sim_cat_it:.3f}')
print('Expected: cat/dog Q similarity > cat/it Q similarity (similar embeddings -> similar projections).')

---

*This notebook accompanies [Chapter 7: Paying Attention](https://learnai.robennals.org/attention). Did you notice the toy attention has no idea where words are in the sentence? [Chapter 8: Where Am I?](https://learnai.robennals.org/positions) fixes that with **rotation**.*

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*